# Roman Strong Lens Data Challenge – Rung 0
## Vision Transformer Regressor – Einstein Radius

This notebook trains a Vision Transformer on simulated Roman images of strong gravitational lenses. Its convolutional stem embeds image patches before a Transformer encoder estimates the Einstein radius (θ_e).

The dataset contains lens systems only. The regressor predicts θ_e for every image without a classification stage.

### Pipeline overview
1. Load the dataset from the `.h5` file
2. Build color images from the three infrared filters (F106, F129, F158)
3. Train the ViT regressor on the lens images
4. Evaluate performance: overall metrics, error vs. SNR, worst predictions, performance by Einstein radius quartile


## Imports and setup

In [ ]:
import math
import h5py
import numpy as np
import matplotlib.pyplot as plt
from astropy.visualization import make_lupton_rgb

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split

from sklearn.metrics import r2_score

# Use a GPU when CUDA is available.
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if device.type == 'cuda':
    print(f'GPU name: {torch.cuda.get_device_name(0)}')


## Load the dataset

In [ ]:
# HDF5 input
h5_filepath = 'roman_data_challenge_rung_0_v_2_0.h5'

f = h5py.File(h5_filepath, 'r')

# List top-level groups to locate the image data.
print('Top-level keys in file:', list(f.keys()))


In [ ]:
images_group = f['images']


In [ ]:
print(f'Number of lens systems: {len(images_group.keys())}')

first_key = list(images_group.keys())[0]
first_group = images_group[first_key]
print(f'\nGroup-level attributes for first system: {list(first_group.attrs.keys())}')
print(f'Datasets inside first system: {list(first_group.keys())}')


In [ ]:
from challenge_data import attr_val

sample_keys = list(images_group.keys())[:10]
for k in sample_keys:
    grp = images_group[k]
    theta_e = float(attr_val(grp.attrs, 'theta_e'))
    substructure = attr_val(grp.attrs, 'substructure')
    print(f"{k}: theta_e={theta_e:.4f}, substructure={substructure}")

## Build image tensors and labels

In [ ]:
images = []
labels = []
rgb_previews = []
snr_values = []

from challenge_data import get_filter_dataset, normalize

for strong_lens_group in images_group.keys():
    group = images_group[strong_lens_group]

    einstein_radius = float(attr_val(group.attrs, 'theta_e'))

    ds_r = get_filter_dataset(group, 'F158')
    ds_g = get_filter_dataset(group, 'F129')
    ds_b = get_filter_dataset(group, 'F106')

    raw_r = ds_r[:]
    raw_g = ds_g[:]
    raw_b = ds_b[:]

    snr = float(attr_val(ds_b.attrs, 'snr'))  # pulled from the F106 exposure

    norm_r = normalize(raw_r)
    norm_g = normalize(raw_g)
    norm_b = normalize(raw_b)

    stacked = np.stack([norm_r, norm_g, norm_b], axis=0)
    images.append(stacked)
    labels.append(einstein_radius)
    snr_values.append(snr)

    rgb = make_lupton_rgb(image_r=norm_r, image_g=norm_g, image_b=norm_b,
                          stretch=0.2, Q=4)
    rgb_previews.append(rgb)

images = np.array(images, dtype=np.float32)
labels = np.array(labels, dtype=np.float32)
snr_values = np.array(snr_values, dtype=np.float32)
IMG_SIZE = images.shape[-1]  # Image height and width
print(f'Image size (H=W)   : {IMG_SIZE}')

print(f'Images array shape : {images.shape}')
print(f'Labels array shape : {labels.shape}')
print(f'SNR range {snr_values.min():.2f} to {snr_values.max():.2f}')

## Preview the image data

In [ ]:
grid_size = 5
_, ax = plt.subplots(grid_size, grid_size, figsize=(12, 12), constrained_layout=True)

for i, (rgb, er) in enumerate(zip(rgb_previews[:grid_size**2], labels[:grid_size**2])):
    row, col = i // grid_size, i % grid_size
    ax[row, col].imshow(rgb)
    ax[row, col].set_title(f'θ_e = {er:.2f}"', fontsize=9)
    ax[row, col].axis('off')

plt.suptitle('Sample Lenses – Lupton RGB (F158/F129/F106)', fontsize=14, y=1.01)
plt.show()

# Also plot the distribution of Einstein radii
plt.figure(figsize=(8, 4))
plt.hist(labels, bins=50, color='steelblue', edgecolor='white')
plt.xlabel('Einstein Radius (arcseconds)')
plt.ylabel('Count')
plt.title('Distribution of Einstein Radii in Dataset')
plt.tight_layout()
plt.show()


## Augment the training images

In [ ]:
from challenge_data import augment_dataset

images_aug, labels_aug = augment_dataset(images, labels)

print(f'Original dataset  : {images.shape[0]:,} images')
print(f'Augmented dataset : {images_aug.shape[0]:,} images  (8× via dihedral symmetry)')

## Build a PyTorch dataset and data loaders

In [ ]:
class LensDataset(Dataset):
    """Simple PyTorch Dataset wrapping our numpy arrays."""
    def __init__(self, images, labels):
        self.images = torch.tensor(images, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.float32)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.images[idx], self.labels[idx]


full_dataset = LensDataset(images_aug, labels_aug)
total = len(full_dataset)

n_train = int(0.70 * total)
n_val   = int(0.15 * total)
n_test  = total - n_train - n_val

train_set, val_set, test_set = random_split(
    full_dataset, [n_train, n_val, n_test],
    generator=torch.Generator().manual_seed(42)   # reproducible split
)

BATCH_SIZE = 64

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_set,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_set,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f'Training samples   : {n_train:,}')
print(f'Validation samples : {n_val:,}')
print(f'Test samples       : {n_test:,}')


## Define the vision transformer

In [ ]:
class PatchEmbedding(nn.Module):
    '''
    Small convolutional stem followed by patchification. The stem applies
    three stride-1 3x3 convolutions before self-attention and preserves the
    image dimensions. The code pads to a patch multiple, then projects each
    patch to the transformer embedding.
    '''
    def __init__(self, in_channels=3, patch_size=9, embed_dim=128,
                 img_size=73, stem_channels=32):
        super().__init__()
        self.patch_size = patch_size

        # ── Convolutional stem: 3 stride-1 conv layers, no downsampling ──
        self.stem = nn.Sequential(
            nn.Conv2d(in_channels, stem_channels, kernel_size=3,
                      stride=1, padding=1, bias=False),
            nn.BatchNorm2d(stem_channels),
            nn.ReLU(inplace=True),

            nn.Conv2d(stem_channels, stem_channels, kernel_size=3,
                      stride=1, padding=1, bias=False),
            nn.BatchNorm2d(stem_channels),
            nn.ReLU(inplace=True),

            nn.Conv2d(stem_channels, stem_channels, kernel_size=3,
                      stride=1, padding=1, bias=False),
            nn.BatchNorm2d(stem_channels),
            nn.ReLU(inplace=True),
        )

        # Compute padded size (next multiple of patch_size >= img_size)
        self.padded_size = math.ceil(img_size / patch_size) * patch_size
        pad_total = self.padded_size - img_size
        self.pad_left = pad_total // 2
        self.pad_right = pad_total - self.pad_left

        self.grid_size = self.padded_size // patch_size
        self.num_patches = self.grid_size * self.grid_size

        # Patchify: projects from stem_channels (not raw 3) to embed_dim
        self.proj = nn.Conv2d(stem_channels, embed_dim,
                               kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        x = self.stem(x)              # (batch, stem_channels, 73, 73)
        x = F.pad(x, (self.pad_left, self.pad_right,
                       self.pad_left, self.pad_right))
        x = self.proj(x)              # (batch, embed_dim, grid, grid)
        x = x.flatten(2)              # (batch, embed_dim, num_patches)
        x = x.transpose(1, 2)         # (batch, num_patches, embed_dim)
        return x


class TransformerNet(nn.Module):
    '''
    Vision Transformer regressor. The final linear layer returns one scalar
    with no activation, which represents the predicted theta_e.

    Architecture:
        Input        : (batch, 3, 91, 91)
        Patch embed  : conv stem -> pad to grid -> 9x9 patches -> embed_dim
        CLS token    : learnable token prepended to the sequence
        Pos embed    : learnable positional embedding added to every token
        Transformer  : `depth` stacked encoder blocks (self-attn + MLP)
        Head         : CLS token's output -> MLP -> scalar
    '''
    def __init__(self, in_channels=3, img_size=73, patch_size=9,
                 embed_dim=96, depth=4, num_heads=6, mlp_ratio=4.0,
                 dropout=0.1):
        super().__init__()

        self.patch_embed = PatchEmbedding(in_channels, patch_size, embed_dim, img_size)
        num_patches = self.patch_embed.num_patches

        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches + 1, embed_dim))
        self.pos_drop = nn.Dropout(dropout)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=int(embed_dim * mlp_ratio),
            dropout=dropout,
            activation='gelu',
            batch_first=True,
            norm_first=True   # pre-LN blocks train more stably than post-LN
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=depth)
        self.norm = nn.LayerNorm(embed_dim)

        self.fc_head = nn.Sequential(
            nn.Linear(embed_dim, 64),
            nn.ReLU(),
            nn.Dropout(p=0.3),
            nn.Linear(64, 1)
        )

        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        nn.init.trunc_normal_(self.cls_token, std=0.02)

    def forward(self, x):
        batch_size = x.shape[0]

        x = self.patch_embed(x)                       # (batch, num_patches, embed_dim)

        cls_tokens = self.cls_token.expand(batch_size, -1, -1)
        x = torch.cat((cls_tokens, x), dim=1)          # (batch, num_patches+1, embed_dim)
        x = x + self.pos_embed
        x = self.pos_drop(x)

        x = self.transformer(x)
        x = self.norm(x)

        cls_output = x[:, 0]                           # take CLS token only
        out = self.fc_head(cls_output)
        return out.squeeze(1)


## Configure training

In [ ]:
## Train the regressor

## Train the classifier

In [ ]:
train_losses = []
val_losses   = []
best_val_loss = float('inf')

early_stopper.best_loss = float('inf')
early_stopper.counter = 0
early_stopper.should_stop = False

for epoch in range(NUM_EPOCHS):

    model.train()
    running_train_loss = 0.0

    for images_batch, labels_batch in train_loader:
        images_batch = images_batch.to(device)
        labels_batch = labels_batch.to(device)

        optimizer.zero_grad()
        predictions = model(images_batch)
        loss = criterion(predictions, labels_batch)
        loss.backward()
        optimizer.step()

        running_train_loss += loss.item() * len(images_batch)

    avg_train_loss = running_train_loss / n_train

    model.eval()
    running_val_loss = 0.0

    with torch.no_grad():
        for images_batch, labels_batch in val_loader:
            images_batch = images_batch.to(device)
            labels_batch = labels_batch.to(device)
            predictions  = model(images_batch)
            loss = criterion(predictions, labels_batch)
            running_val_loss += loss.item() * len(images_batch)

    avg_val_loss = running_val_loss / n_val

    train_losses.append(avg_train_loss)
    val_losses.append(avg_val_loss)

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), 'best_model_vit_lensonly.pt')

    old_lr = optimizer.param_groups[0]['lr']
    if epoch < WARMUP_EPOCHS:
        warmup_scheduler.step()
    else:
        scheduler.step(avg_val_loss)
    new_lr = optimizer.param_groups[0]['lr']
    if new_lr != old_lr:
        tag = 'Warmup' if epoch < WARMUP_EPOCHS else 'Learning rate reduced'
        print(f'  → {tag}: {old_lr:.2e} → {new_lr:.2e}')

    if (epoch + 1) % 5 == 0:
        train_rmse = np.sqrt(avg_train_loss)
        val_rmse   = np.sqrt(avg_val_loss)
        print(f'Epoch {epoch+1:3d}/{NUM_EPOCHS} – '
              f'Train RMSE: {train_rmse:.4f}"  |  '
              f'Val RMSE: {val_rmse:.4f}"  |  '
              f'EarlyStop counter: {early_stopper.counter}/{early_stopper.patience}')

    if early_stopper.step(avg_val_loss):
        print(f'\nEarly stopping triggered at epoch {epoch+1} '
              f'(no improvement for {early_stopper.patience} epochs).')
        print(f'Best validation loss (MSE): {early_stopper.best_loss:.6f}')
        break

print(f'\nBest validation loss (MSE): {best_val_loss:.6f}')
print(f'Best validation RMSE      : {np.sqrt(best_val_loss):.4f} arcseconds')


## Plot training history

In [ ]:
epochs = range(1, epoch+2)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(epochs, train_losses, label='Train Loss (Huber)', color='steelblue')
axes[0].plot(epochs, val_losses,   label='Val Loss (Huber)',   color='tomato')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training vs Validation Loss')
axes[0].legend()

axes[1].plot(epochs, np.sqrt(train_losses), label='Train RMSE', color='steelblue')
axes[1].plot(epochs, np.sqrt(val_losses),   label='Val RMSE',   color='tomato')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('RMSE (arcseconds)')
axes[1].set_title('Training vs Validation RMSE')
axes[1].legend()

plt.tight_layout()
plt.show()


## Evaluate on held-out test images

In [ ]:
model.load_state_dict(torch.load('best_model_vit_lensonly.pt', map_location=device))
model.eval()

all_preds  = []
all_labels = []

with torch.no_grad():
    for images_batch, labels_batch in test_loader:
        images_batch = images_batch.to(device)
        preds = model(images_batch)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels_batch.numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)

test_mse  = np.mean((all_preds - all_labels) ** 2)
test_rmse = np.sqrt(test_mse)
test_mae  = np.mean(np.abs(all_preds - all_labels))
test_r2   = r2_score(all_labels, all_preds)

print('── Test Set Results ──────────────────────────────')
print(f'RMSE : {test_rmse:.4f} arcseconds')
print(f'MAE  : {test_mae:.4f} arcseconds')
print(f'R²   : {test_r2:.4f}')
print('──────────────────────────────────────────────────')
print()
print('Interpretation:')
print(f'  On average, predictions are off by ~{test_rmse:.3f} arcseconds')
print(f'  R² of {test_r2:.3f} means the model explains {test_r2*100:.1f}% of the variance in Einstein radius')


## Visualize the model predictions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].scatter(all_labels, all_preds, alpha=0.3, s=8, color='steelblue')
lims = [all_labels.min(), all_labels.max()]
axes[0].plot(lims, lims, 'r--', linewidth=1.5, label='Perfect prediction')
axes[0].set_xlabel('True Einstein Radius (arcseconds)')
axes[0].set_ylabel('Predicted Einstein Radius (arcseconds)')
axes[0].set_title(f'Predicted vs True  (R² = {test_r2:.3f})')
axes[0].legend()

residuals = all_preds - all_labels
axes[1].scatter(all_labels, residuals, alpha=0.3, s=8, color='steelblue')
axes[1].axhline(0, color='red', linestyle='--', linewidth=1.5)
axes[1].set_xlabel('True Einstein Radius (arcseconds)')
axes[1].set_ylabel('Residual: Predicted − True (arcseconds)')
axes[1].set_title('Residuals vs True Einstein Radius')

plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 4))
plt.hist(residuals, bins=60, color='steelblue', edgecolor='white')
plt.axvline(0, color='red', linestyle='--')
plt.xlabel('Residual (arcseconds)')
plt.ylabel('Count')
plt.title('Distribution of Prediction Residuals')
plt.tight_layout()
plt.show()


In [ ]:
# ── Error vs SNR ─────────────────────────────────────────────────────────────
test_indices = test_set.indices
orig_indices_full = np.array([idx // 8 for idx in test_indices])

test_snr = snr_values[orig_indices_full]
abs_error = np.abs(all_preds - all_labels)

theta_e_median = np.median(all_labels)
is_large_theta_e = all_labels >= theta_e_median

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

sc = axes[0].scatter(test_snr, abs_error, c=all_labels, cmap='viridis',
                      alpha=0.5, s=12)
axes[0].set_xlabel('Signal-to-Noise Ratio')
axes[0].set_ylabel('Absolute Error (arcsec)')
axes[0].set_xscale('log')
axes[0].set_title('Error vs SNR, colored by true θ_e')
cbar = plt.colorbar(sc, ax=axes[0])
cbar.set_label('True θ_e (arcsec)')

n_bins = 8
snr_bin_edges = np.percentile(test_snr, np.linspace(0, 100, n_bins + 1))
snr_bin_centers = 0.5 * (snr_bin_edges[:-1] + snr_bin_edges[1:])

for mask, label, color in [(~is_large_theta_e, f'Small θ_e (< {theta_e_median:.2f}")', 'steelblue'),
                             (is_large_theta_e, f'Large θ_e (≥ {theta_e_median:.2f}")', 'crimson')]:
    binned_median_error = []
    for i in range(n_bins):
        in_bin = mask & (test_snr >= snr_bin_edges[i]) & (test_snr < snr_bin_edges[i + 1])
        if in_bin.sum() > 0:
            binned_median_error.append(np.median(abs_error[in_bin]))
        else:
            binned_median_error.append(np.nan)
    axes[1].plot(snr_bin_centers, binned_median_error, 'o-', color=color, label=label)

axes[1].set_xlabel('Signal-to-Noise Ratio (binned)')
axes[1].set_ylabel('Median Absolute Error (arcsec)')
axes[1].set_xscale('log')
axes[1].set_title('Median absolute error by SNR')
axes[1].legend()

plt.tight_layout()
plt.show()


In [ ]:
from challenge_evaluation import (
    attach_physical_properties,
    get_worst_predictions,
    plot_worst_images,
    print_worst_summary,
)

In [ ]:
worst_df = get_worst_predictions(all_preds, all_labels, orig_indices_full, n_worst=25)
worst_df = attach_physical_properties(worst_df, labels, snr_values=snr_values)
print_worst_summary(worst_df, n_show=25)
plot_worst_images(worst_df, rgb_previews, n_show=12, cols=3, panel_size=5)


In [ ]:
theta_e_bin_edges = np.percentile(all_labels, [0, 25, 50, 75, 100])
theta_e_bin_edges[-1] += 0.01
theta_e_bin_labels = ['Q1 (smallest θ_e)', 'Q2', 'Q3', 'Q4 (largest θ_e)']

print(f'Test set θ_e range: {all_labels.min():.3f}" to {all_labels.max():.3f}"')
print(f'Quartile edges: {theta_e_bin_edges}')

print('\n── R² and Error Metrics by Einstein Radius Quartile ────────────────────')
print(f'{"Bin":<20} {"Range (arcsec)":<20} {"N":>6} {"RMSE":>8} {"MAE":>8} {"R²":>8}')
print('─' * 75)

bin_results = []
for i in range(len(theta_e_bin_edges) - 1):
    lo, hi = theta_e_bin_edges[i], theta_e_bin_edges[i + 1]
    mask = (all_labels >= lo) & (all_labels < hi)
    n_in_bin = mask.sum()

    if n_in_bin < 5:
        print(f'{theta_e_bin_labels[i]:<20} {"":<20} {n_in_bin:>6}    (too few samples)')
        continue

    bin_preds  = all_preds[mask]
    bin_truth  = all_labels[mask]

    bin_rmse = np.sqrt(np.mean((bin_preds - bin_truth) ** 2))
    bin_mae  = np.mean(np.abs(bin_preds - bin_truth))
    bin_r2   = r2_score(bin_truth, bin_preds)

    range_str = f'{lo:.3f}–{hi:.3f}'
    bin_results.append({
        'bin': theta_e_bin_labels[i], 'range': range_str, 'n': n_in_bin,
        'rmse': bin_rmse, 'mae': bin_mae, 'r2': bin_r2
    })

    print(f'{theta_e_bin_labels[i]:<20} {range_str:<20} {n_in_bin:>6} {bin_rmse:>8.4f} {bin_mae:>8.4f} {bin_r2:>8.4f}')

print('─' * 75)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

bins_plotted = [r['bin']  for r in bin_results]
r2_plotted   = [r['r2']   for r in bin_results]
rmse_plotted = [r['rmse'] for r in bin_results]
mae_plotted  = [r['mae']  for r in bin_results]

axes[0].bar(bins_plotted, r2_plotted, color='steelblue', edgecolor='white')
axes[0].set_xlabel('Einstein Radius Quartile')
axes[0].set_ylabel('R²')
axes[0].set_title('R² by Einstein Radius (θ_e) Quartile')
axes[0].tick_params(axis='x', rotation=15)

x = np.arange(len(bins_plotted))
width = 0.35
axes[1].bar(x - width/2, rmse_plotted, width, label='RMSE', color='tomato')
axes[1].bar(x + width/2, mae_plotted,  width, label='MAE',  color='steelblue')
axes[1].set_xticks(x)
axes[1].set_xticklabels(bins_plotted, rotation=15)
axes[1].set_xlabel('Einstein Radius Quartile')
axes[1].set_ylabel('Error (arcseconds)')
axes[1].set_title('RMSE vs MAE by Einstein Radius (θ_e) Quartile')
axes[1].legend()

plt.tight_layout()
plt.show()

## Inspect individual predictions

In [ ]:
test_indices = test_set.indices
orig_indices = [idx // 8 for idx in test_indices[:25]]

fig, axes = plt.subplots(5, 5, figsize=(14, 14), constrained_layout=True)

for i, (orig_idx, pred, true) in enumerate(zip(orig_indices, all_preds[:25], all_labels[:25])):
    row, col = i // 5, i % 5
    axes[row, col].imshow(rgb_previews[orig_idx])
    axes[row, col].set_title(
        f'True: {true:.2f}"\nPred: {pred:.2f}"',
        fontsize=8,
        color='green' if abs(pred - true) < 0.05 else 'red'
    )
    axes[row, col].axis('off')

plt.suptitle('Test Set Predictions  (green = within 0.05", red = larger error)', fontsize=13)
plt.show()


In [ ]:
THRESHOLD = 0.05

correct = np.abs(all_preds - all_labels) < THRESHOLD
pct_correct = 100 * np.mean(correct)

print(f'Correct (within {THRESHOLD}"): {correct.sum()} / {len(all_labels)}')
print(f'Percentage correct: {pct_correct:.2f}%')


---
## Summary

`TransformerNet` uses a convolutional stem, patch embedding, and a Transformer encoder to estimate θ_e directly. The training code warms up the learning rate, then adjusts it using validation loss. It saves the checkpoint with the lowest validation loss as `best_model_vit_lensonly.pt`.

Run the cells from top to bottom. Training may take a while. The evaluation cells load the saved checkpoint.

To inspect errors, compare absolute error across SNR bins and look for clusters by θ_e or SNR in the worst-predictions table.
